# World Foresight Framework — Data Extract & Transform

Builds the `Historical Data` sheet of `Final Data.xlsx`, one proxy at a time.

Every proxy follows the same four steps:

1. **Extract** — read the source (a downloaded file, or the World Bank API) and
   show `head(5)` so you can see what came back.
2. **Transform** — reshape it into the six-column contract.
3. **`check_coverage()`** — how many of the 34 markets are covered, and which
   are missing.
4. **`load()`** — append the result to the workbook.

| Column | Meaning |
|--------|---------|
| `id` | Proxy dimension, e.g. `D1` or `D7_1` (derived on load) |
| `proxy_id` | `D{n}_{ISO3}`, e.g. `D1_USA` |
| `market` | ISO3 country code, or `GLO` for a global series |
| `year` | Integer year, 2000–2025 |
| `value` | Transformed value |
| `labels` | Display unit, e.g. `% of GDP` |
| `metric` | Scale already applied, e.g. `MILLIONS` |

Shared helpers live in `Transform_Functions/common.py`, `Transform_Functions/excel_io.py`
and `Extract_Functions/world_bank.py`.

## Setup

In [2]:
# Run from the project root or from "Data Preparation"; both work.
from pathlib import Path

_here = Path.cwd().resolve()
DATA_PREP_DIR = _here if _here.name == "Data Preparation" else _here / "Data Preparation"
if not DATA_PREP_DIR.is_dir():
    raise FileNotFoundError("Open this notebook from the project root or Data Preparation")

%cd {DATA_PREP_DIR}
%reload_ext autoreload
%autoreload 2

/Users/dunglai/Desktop/Việt Dũng/Personal Projects/World Foresight Framework/Data Preparation


In [3]:
import json

import numpy as np
import pandas as pd

import Extract_Functions.world_bank as wb
from Transform_Functions.common import (
    END_YEAR,
    MARKETS,
    RAW_DIR,
    START_YEAR,
    melt_years,
    read_prepared,
    to_contract,
    to_iso3,
)
from Transform_Functions.excel_io import check_coverage, load

print(f"{len(MARKETS)} markets, {START_YEAR}-{END_YEAR}")

34 markets, 2000-2025


### Repeated patterns

Helpers for the two proxy families that repeat the same shape many times over.

In [4]:
UN_MEMBERS = 193

def treaty_uptake(parties, proxy_id, un_members=UN_MEMBERS):
    """Cumulative share of UN member states party to a treaty, by year end.

    Years before the treaty had any party are dropped rather than recorded as 0.
    """
    # Parties that are not UN member states cannot count toward a share of 193.
    NON_UN = {"european union", "eu", "holy see", "holy see (vatican city state)",
              "vatican city", "vatican", "state of palestine", "palestine",
              "cook islands", "niue"}

    names = parties["country"].astype(str).str.replace(r"\[[^\]]*\]", "", regex=True)
    names = names.str.replace("†", "", regex=False).str.replace("*", "", regex=False)
    names = names.str.strip().str.lower()

    joined = pd.to_numeric(parties.loc[~names.isin(NON_UN), "party_date"]
                           .astype(str).str[:4], errors="coerce").dropna()

    years = range(START_YEAR, END_YEAR + 1)
    counts = pd.DataFrame({"year": years})
    counts["parties"] = counts["year"].map(lambda y: int((joined <= y).sum()))
    counts = counts[counts["parties"] > 0]
    counts["value"] = counts["parties"] / un_members * 100
    counts["market"] = "GLO"

    return to_contract(counts, proxy_id, labels="%", decimals=2)


def vdem_country_year():
    """Load the V-Dem country-year file once for the six D49-D54 proxies."""
    return pd.read_csv(RAW_DIR / "V-Dem" / "vdem_country_year.csv", low_memory=False)


def vdem_indicator(raw, proxy_id, variable):
    """Pull one V-Dem variable for the 34 markets.

    `country_text_id` is already ISO3 in V-Dem, so no name mapping is needed.
    """
    data = raw[["country_text_id", "year", variable]].rename(
        columns={"country_text_id": "market", variable: "value"})
    return to_contract(data[data["market"].isin(MARKETS)], proxy_id, labels="Index")

# Topic 1: Power

## D1

In [ ]:
# Row 6 holds the header: Country, Notes, 1949, 1950, ...
raw_d1 = pd.read_excel(RAW_DIR / "SIPRI-Milex-data-1949-2025_v1.2.xlsx",
                       sheet_name="Current US$", header=5)

raw_d1.head(5)

In [ ]:
# SIPRI marks unavailable years with ".." / "xxx"; to_numeric turns those into
# NaN and to_contract drops them. Values are already USD millions.
d1 = melt_years(raw_d1, "Country")
d1["market"] = to_iso3(d1["Country"])

df_d1 = to_contract(d1, "D1", labels="USD (millions)", metric="MILLIONS")
df_d1.head(5)

In [ ]:
check_coverage(df_d1)

In [ ]:
load(df_d1)

## D2

In [ ]:
raw_d2 = wb.fetch_many(["NE.EXP.GNFS.CD", "NE.IMP.GNFS.CD"])

raw_d2.head(5)

In [ ]:
# Total trade = exports + imports of goods and services.
d2 = raw_d2.dropna(subset=["NE.EXP.GNFS.CD", "NE.IMP.GNFS.CD"], how="all").copy()
d2["value"] = d2[["NE.EXP.GNFS.CD", "NE.IMP.GNFS.CD"]].sum(axis=1)
d2 = d2[d2["value"] > 0]

df_d2 = to_contract(d2, "D2", labels="USD (billions)", metric="BILLIONS", scale=1e9)
df_d2.head(5)

In [ ]:
check_coverage(df_d2)

In [ ]:
load(df_d2)

## D3

In [ ]:
raw_d3 = pd.read_csv(RAW_DIR / "D3_soft_power_scores.csv")

raw_d3.head(5)

In [ ]:
# This export already carries market codes; `date` holds the survey year.
df_d3 = to_contract(raw_d3, "D3", labels="Index", year_col="date")
df_d3.head(5)

In [ ]:
check_coverage(df_d3)

In [ ]:
load(df_d3)

## D4

In [ ]:
raw_d4 = wb.fetch("NY.GDP.MKTP.CD")

raw_d4.head(5)

In [ ]:
df_d4 = to_contract(raw_d4[raw_d4["value"] > 0], "D4",
                    labels="USD (trillions)", metric="TRILLIONS", scale=1e12)
df_d4.head(5)

In [ ]:
check_coverage(df_d4)

In [ ]:
load(df_d4)

## D5

In [ ]:
# -9 is the COW missing-value marker.
raw_d5 = pd.read_csv(RAW_DIR / "NMCv7" / "NMC-70-abridged.csv", na_values=[-9, "-9"])

raw_d5.head(5)

In [ ]:
# COW state abbreviations are not ISO3 (AUL=Australia, GMY=Germany, DRV=Vietnam),
# so they need their own lookup rather than the shared to_iso3 aliases.
COW_TO_ISO3 = {
    "USA": "USA", "CAN": "CAN", "MEX": "MEX", "BRA": "BRA", "ARG": "ARG",
    "GMY": "DEU", "FRN": "FRA", "UKG": "GBR", "ITA": "ITA", "RUS": "RUS",
    "TUR": "TUR", "POL": "POL", "NTH": "NLD", "UKR": "UKR", "CHN": "CHN",
    "JPN": "JPN", "ROK": "KOR", "INS": "IDN", "AUL": "AUS", "DRV": "VNM",
    "IND": "IND", "PAK": "PAK", "BNG": "BGD", "SAU": "SAU", "UAE": "ARE",
    "IRN": "IRN", "ISR": "ISR", "EGY": "EGY", "NIG": "NGA", "SAF": "ZAF",
    "ETH": "ETH", "KEN": "KEN", "DRC": "COD", "KZK": "KAZ",
}

d5 = raw_d5[["stateabb", "year", "cinc"]].copy()
d5["market"] = d5["stateabb"].str.strip().map(COW_TO_ISO3)
d5["value"] = pd.to_numeric(d5["cinc"], errors="coerce")
d5 = d5.dropna(subset=["market", "value"])
d5 = d5[d5["value"].between(0, 1)]          # CINC is a world share, so 0-1

df_d5 = to_contract(d5, "D5", labels="CINC score", decimals=6)
df_d5.head(5)

In [ ]:
check_coverage(df_d5)

In [ ]:
load(df_d5)

## D6

In [ ]:
raw_d6 = pd.read_csv(RAW_DIR / "Idealpointestimates1946-2025.csv")

raw_d6.head(5)

In [ ]:
# The file is already keyed by ISO3 (`iso3c`), so no name mapping is needed.
d6 = raw_d6.rename(columns={"iso3c": "market", "IdealPointFP": "value"})

df_d6 = to_contract(d6[d6["market"].isin(MARKETS)], "D6", labels="Index", decimals=6)
df_d6.head(5)

In [ ]:
check_coverage(df_d6)

In [ ]:
load(df_d6)

## D7_1, D7_2, D7_3, D7_4

In [ ]:
# The file carries 11 lines of licence preamble before the header.
raw_d7 = pd.read_csv(RAW_DIR / "trade-register.csv", encoding="latin1",
                     skiprows=11, on_bad_lines="skip")
raw_d7.columns = raw_d7.columns.str.strip()

raw_d7.head(5)

In [ ]:
# Four sub-proxies from one 5-year rolling window of delivered SIPRI TIV:
#   D7_1 supplier concentration (HHI x 100)   D7_3 Eastern-bloc supplier share
#   D7_2 Western supplier share               D7_4 all other suppliers
WINDOW = 5
MIN_WINDOW_TIV = 1.0        # skip country-windows with negligible imports

EASTERN = {"Russia", "China", "North Korea", "Iran", "Belarus"}
WESTERN = {
    "United States", "United Kingdom", "France", "Germany", "Italy",
    "Netherlands", "Spain", "Sweden", "Canada", "Norway", "Switzerland",
    "Poland", "Australia", "Czechia", "Denmark", "Belgium", "Finland",
    "Portugal", "Bulgaria", "Slovakia", "Romania", "Ireland", "Austria",
    "Croatia", "Lithuania", "Latvia", "Estonia", "Slovenia", "Greece",
    "Hungary", "Cyprus", "Malta", "North Macedonia", "Montenegro",
    "Bosnia-Herzegovina", "Turkiye", "Japan", "South Korea", "New Zealand",
    "Israel",
}

def delivery_years(text):
    """'2018; 2019' -> [2018, 2019];  '2017; ?' -> [2017];  '?' -> []."""
    years = []
    for part in str(text).split(";"):
        part = part.strip().lstrip("?").strip()
        if part.isdigit() and len(part) == 4:
            years.append(int(part))
    return years

d7 = raw_d7[["Recipient", "Supplier", "Year(s) of delivery",
             "SIPRI TIV of delivered weapons"]].copy()
d7.columns = ["recipient", "supplier", "years", "tiv"]
d7["tiv"] = pd.to_numeric(d7["tiv"], errors="coerce")
d7["market"] = to_iso3(d7["recipient"])
d7 = d7.dropna(subset=["tiv", "market", "supplier"])

# One order can be delivered across several years; spread its TIV evenly.
spread = []
for row in d7.itertuples():
    years = delivery_years(row.years)
    if years:
        share = row.tiv / len(years)
        spread += [{"market": row.market, "supplier": row.supplier,
                    "year": y, "tiv": share} for y in years]
grid = pd.DataFrame(spread).groupby(
    ["market", "supplier", "year"], as_index=False)["tiv"].sum()

rows = []
for market, group in grid.groupby("market"):
    for year in range(START_YEAR, END_YEAR + 1):
        window = group[group["year"].between(year - WINDOW + 1, year)]
        total = window["tiv"].sum()
        if total < MIN_WINDOW_TIV:
            continue
        by_supplier = window.groupby("supplier")["tiv"].sum()
        west = by_supplier[by_supplier.index.isin(WESTERN)].sum() / total
        east = by_supplier[by_supplier.index.isin(EASTERN)].sum() / total
        rows.append({
            "market": market, "year": year,
            "D7_1": ((by_supplier / total) ** 2).sum() * 100,
            "D7_2": west * 100,
            "D7_3": east * 100,
            "D7_4": max(0.0, 1.0 - west - east) * 100,   # residual
        })
wide_d7 = pd.DataFrame(rows)

df_d7 = pd.concat([
    to_contract(wide_d7, "D7_1", labels="HHI Index", value_col="D7_1", decimals=6),
    to_contract(wide_d7, "D7_2", labels="%", value_col="D7_2", decimals=6),
    to_contract(wide_d7, "D7_3", labels="%", value_col="D7_3", decimals=6),
    to_contract(wide_d7, "D7_4", labels="%", value_col="D7_4", decimals=6),
], ignore_index=True)
df_d7.head(5)

In [ ]:
check_coverage(df_d7)

In [ ]:
load(df_d7)

## D8_1, D8_2, D8_3, D8_4

In [ ]:
raw_d8 = pd.read_csv(RAW_DIR / "AgreementScores.csv",
                     usecols=["ccode1", "ccode2", "agree", "year"])

raw_d8.head(5)

In [ ]:
# Share of votes cast the same way as each major power:
#   D8_1 vs USA    D8_2 vs China    D8_3 vs Russia    D8_4 vs India
# Germany has two historical COW codes (255 unified, 260 West); both map to DEU.
CCODE_TO_ISO3 = {
    2: "USA", 20: "CAN", 70: "MEX", 140: "BRA", 160: "ARG",
    255: "DEU", 260: "DEU", 220: "FRA", 200: "GBR", 325: "ITA",
    365: "RUS", 640: "TUR", 290: "POL", 210: "NLD", 369: "UKR",
    710: "CHN", 740: "JPN", 732: "KOR", 850: "IDN", 900: "AUS",
    816: "VNM", 750: "IND", 770: "PAK", 771: "BGD", 670: "SAU",
    696: "ARE", 630: "IRN", 666: "ISR", 651: "EGY", 475: "NGA",
    560: "ZAF", 530: "ETH", 501: "KEN", 490: "COD", 705: "KAZ",
}
ANCHORS_D8 = {"USA": 1, "CHN": 2, "RUS": 3, "IND": 4}

d8 = raw_d8.copy()
d8["agree"] = pd.to_numeric(d8["agree"], errors="coerce")
d8 = d8.dropna(subset=["agree"])

parts = []
for anchor, index in ANCHORS_D8.items():
    code = next(c for c, iso in CCODE_TO_ISO3.items() if iso == anchor)
    # The anchor can sit on either side of the dyad; the partner is the other.
    left = d8[d8["ccode1"] == code].assign(market=lambda x: x["ccode2"].map(CCODE_TO_ISO3))
    right = d8[d8["ccode2"] == code].assign(market=lambda x: x["ccode1"].map(CCODE_TO_ISO3))
    pair = pd.concat([left, right], ignore_index=True)
    pair = pair[pair["market"].isin(MARKETS) & pair["market"].ne(anchor)]
    # The file stores both directed orderings, giving identical mirrored rows.
    pair = pair.drop_duplicates(["market", "year"])
    pair["value"] = pair["agree"] * 100
    parts.append(to_contract(pair, f"D8_{index}", labels="%"))

df_d8 = pd.concat(parts, ignore_index=True)
df_d8.head(5)

In [ ]:
check_coverage(df_d8)

In [ ]:
load(df_d8)

## D9_1, D9_2, D9_3, D9_4, D9_5

In [ ]:
# Comtrade caps rows per query, so the pull is split across three files.
raw_d9 = pd.concat([
    pd.read_csv(RAW_DIR / name, encoding="latin1", low_memory=False)
    for name in ["TradeData_7_6_2026_0_22_12.csv",
                 "TradeData_7_6_2026_0_22_57.csv",
                 "TradeData_7_6_2026_0_23_25.csv"]
], ignore_index=True)

raw_d9.head(5)

In [ ]:
# Share of total trade (imports + exports) flowing to each anchor power:
#   D9_1 USA   D9_2 China   D9_3 Russia   D9_4 India   D9_5 rest of world
WORLD = "W00"
ANCHORS_D9 = {"USA": 1, "CHN": 2, "RUS": 3, "IND": 4}

d9 = raw_d9[["reporterCode", "refPeriodId", "partnerCode", "fobvalue"]].copy()
d9.columns = ["market", "year", "partner", "value"]
d9["value"] = pd.to_numeric(d9["value"], errors="coerce")
d9["year"] = pd.to_numeric(d9["year"], errors="coerce")
d9 = d9.dropna(subset=["value"])
d9 = d9[d9["market"].isin(MARKETS) & d9["partner"].isin([WORLD, *ANCHORS_D9])]

# Sum both flows per reporter-year-partner, then widen so shares are one divide.
wide_d9 = d9.pivot_table(index=["market", "year"], columns="partner",
                         values="value", aggfunc="sum").reset_index()
present = [a for a in ANCHORS_D9 if a in wide_d9.columns]
wide_d9 = wide_d9[wide_d9[WORLD].notna() & wide_d9[WORLD].ne(0)]
wide_d9[present] = wide_d9[present].fillna(0.0)

parts = []
for anchor in present:
    share = wide_d9.assign(value=wide_d9[anchor] / wide_d9[WORLD] * 100)
    parts.append(to_contract(share, f"D9_{ANCHORS_D9[anchor]}", labels="%", decimals=2))

# Others is the residual, so the five sub-proxies always sum to 100.
others = wide_d9.assign(
    value=((wide_d9[WORLD] - wide_d9[present].sum(axis=1)) / wide_d9[WORLD] * 100).clip(0, 100))
parts.append(to_contract(others, "D9_5", labels="%", decimals=2))

df_d9 = pd.concat(parts, ignore_index=True)
df_d9.head(5)

In [ ]:
check_coverage(df_d9)

In [ ]:
load(df_d9)

## D10

In [ ]:
# Deposit year of the ICJ Article 36(2) declaration currently in force, per
# state, from icj-cij.org/declarations.
ICJ_DEPOSIT_YEARS = [
    2002, 1971, 1980, 1958, 1970, 2015, 1957, 1994, 2023, 1973, 2001, 2002,
    1989, 1956, 2005, 2006, 1924, 1957, 2017, 1991, 1958, 1966, 1995, 2025,
    2015, 1998, 1989, 1921, 1986, 1992, 2026, 2019, 2023, 2011, 2014, 2015,
    2019, 2000, 1952, 1950, 2012, 1930, 1992, 1966, 1983, 2013, 1968, 1947,
    2017, 1977, 1946, 1998, 1996, 2017, 1921, 1996, 2003, 1972, 2024, 2005,
    2015, 1985, 2004, 1963, 1990, 1958, 1987, 1969, 1957, 1948, 2012, 1979,
    1963, 2017, 1921,
]

len(ICJ_DEPOSIT_YEARS)

In [ ]:
# Cumulative share of the 193 UN member states accepting ICJ jurisdiction,
# counted at each year end.
UN_MEMBERS = 193

d10 = pd.DataFrame({"year": range(START_YEAR, END_YEAR + 1)})
d10["value"] = d10["year"].map(
    lambda y: sum(1 for deposit in ICJ_DEPOSIT_YEARS if deposit <= y) / UN_MEMBERS * 100)
d10["market"] = "GLO"

df_d10 = to_contract(d10, "D10", labels="%")
df_d10.head(5)

In [ ]:
check_coverage(df_d10)

In [ ]:
load(df_d10)

## D11_1

In [ ]:
raw_d11_1 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_1_paris.csv")

raw_d11_1.head(5)

In [ ]:
df_d11_1 = treaty_uptake(raw_d11_1, "D11_1")
df_d11_1.head(5)

In [ ]:
check_coverage(df_d11_1)

In [ ]:
load(df_d11_1)

## D11_2

In [ ]:
raw_d11_2 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_2_cbd.csv")

raw_d11_2.head(5)

In [ ]:
df_d11_2 = treaty_uptake(raw_d11_2, "D11_2")
df_d11_2.head(5)

In [ ]:
check_coverage(df_d11_2)

In [ ]:
load(df_d11_2)

## D11_3

In [ ]:
raw_d11_3 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_3_unclos.csv")

raw_d11_3.head(5)

In [ ]:
df_d11_3 = treaty_uptake(raw_d11_3, "D11_3")
df_d11_3.head(5)

In [ ]:
check_coverage(df_d11_3)

In [ ]:
load(df_d11_3)

## D11_4

In [ ]:
raw_d11_4 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_4_rome.csv")

raw_d11_4.head(5)

In [ ]:
df_d11_4 = treaty_uptake(raw_d11_4, "D11_4")
df_d11_4.head(5)

In [ ]:
check_coverage(df_d11_4)

In [ ]:
load(df_d11_4)

## D11_5

In [ ]:
raw_d11_5 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_5_iccpr.csv")

raw_d11_5.head(5)

In [ ]:
df_d11_5 = treaty_uptake(raw_d11_5, "D11_5")
df_d11_5.head(5)

In [ ]:
check_coverage(df_d11_5)

In [ ]:
load(df_d11_5)

## D11_6

In [ ]:
raw_d11_6 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_6_npt.csv")

raw_d11_6.head(5)

In [ ]:
df_d11_6 = treaty_uptake(raw_d11_6, "D11_6")
df_d11_6.head(5)

In [ ]:
check_coverage(df_d11_6)

In [ ]:
load(df_d11_6)

## D11_7

In [ ]:
raw_d11_7 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_7_cwc.csv")

raw_d11_7.head(5)

In [ ]:
df_d11_7 = treaty_uptake(raw_d11_7, "D11_7")
df_d11_7.head(5)

In [ ]:
check_coverage(df_d11_7)

In [ ]:
load(df_d11_7)

## D11_8

In [ ]:
raw_d11_8 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_8_ctbt.csv")

raw_d11_8.head(5)

In [ ]:
df_d11_8 = treaty_uptake(raw_d11_8, "D11_8")
df_d11_8.head(5)

In [ ]:
check_coverage(df_d11_8)

In [ ]:
load(df_d11_8)

## D11_9

In [ ]:
raw_d11_9 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_9_att.csv")

raw_d11_9.head(5)

In [ ]:
df_d11_9 = treaty_uptake(raw_d11_9, "D11_9")
df_d11_9.head(5)

In [ ]:
check_coverage(df_d11_9)

In [ ]:
load(df_d11_9)

## D11_10

In [ ]:
raw_d11_10 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_10_untoc.csv")

raw_d11_10.head(5)

In [ ]:
df_d11_10 = treaty_uptake(raw_d11_10, "D11_10")
df_d11_10.head(5)

In [ ]:
check_coverage(df_d11_10)

In [ ]:
load(df_d11_10)

## D11_11

In [ ]:
raw_d11_11 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_11_uncac.csv")

raw_d11_11.head(5)

In [ ]:
df_d11_11 = treaty_uptake(raw_d11_11, "D11_11")
df_d11_11.head(5)

In [ ]:
check_coverage(df_d11_11)

In [ ]:
load(df_d11_11)

## D11_12

In [ ]:
raw_d11_12 = pd.read_csv(RAW_DIR / "UN_Treaties" / "d11_12_geneva.csv")

raw_d11_12.head(5)

In [ ]:
df_d11_12 = treaty_uptake(raw_d11_12, "D11_12")
df_d11_12.head(5)

In [ ]:
check_coverage(df_d11_12)

In [ ]:
load(df_d11_12)

## D12

In [ ]:
raw_d12 = pd.read_csv(RAW_DIR / "DPO-UCHISTORICAL.csv")

raw_d12.head(5)

In [ ]:
# Annual personnel-months = sum of the twelve monthly headcounts.
d12 = raw_d12.copy()
d12["year"] = pd.to_datetime(d12["last_reporting_date"]).dt.year
d12["personnel"] = d12["male_personnel"].fillna(0) + d12["female_personnel"].fillna(0)
d12 = d12[d12["isocode3"].isin(MARKETS)]

annual = (d12.groupby(["isocode3", "year"], as_index=False)["personnel"].sum()
          .rename(columns={"isocode3": "market", "personnel": "value"}))

# A contributor with no deployment in a reported year is a real zero, so fill
# it -- but only across the years the source covers, since "no data" is not 0.
covered = range(int(annual["year"].min()), int(annual["year"].max()) + 1)
full = pd.MultiIndex.from_product([list(MARKETS), covered],
                                  names=["market", "year"]).to_frame(index=False)
annual = full.merge(annual, on=["market", "year"], how="left").fillna({"value": 0.0})

# GLO is the combined contribution of all 34 project markets.
world = annual.groupby("year", as_index=False)["value"].sum().assign(market="GLO")

df_d12 = to_contract(pd.concat([annual, world], ignore_index=True), "D12",
                     labels="No. of people", decimals=0)
df_d12.head(5)

In [ ]:
check_coverage(df_d12)

In [ ]:
load(df_d12)

## D13

In [ ]:
# Fuel/metal/food exports are published as shares of merchandise exports, so
# the level has to be reconstructed from four indicators.
raw_d13 = wb.fetch_many(["TX.VAL.MRCH.CD.WT", "TX.VAL.FUEL.ZS.UN",
                         "TX.VAL.MMTL.ZS.UN", "TX.VAL.FOOD.ZS.UN"])

raw_d13.head(5)

In [ ]:
SHARES_D13 = ["TX.VAL.FUEL.ZS.UN", "TX.VAL.MMTL.ZS.UN", "TX.VAL.FOOD.ZS.UN"]

d13 = raw_d13.dropna(subset=["TX.VAL.MRCH.CD.WT"]).dropna(subset=SHARES_D13, how="all")
d13["value"] = d13["TX.VAL.MRCH.CD.WT"] * d13[SHARES_D13].sum(axis=1) / 100

df_d13 = to_contract(d13[d13["value"] > 0], "D13",
                     labels="USD (billions)", metric="BILLIONS", scale=1e9, decimals=6)
df_d13.head(5)

In [ ]:
check_coverage(df_d13)

In [ ]:
load(df_d13)

## D14

In [ ]:
raw_d14 = pd.read_csv(RAW_DIR / "AgreementScores.csv")
ideal_points = pd.read_csv(RAW_DIR / "Idealpointestimates1946-2025.csv",
                           usecols=["ccode", "iso3c"])

raw_d14.head(5)

In [ ]:
# Mean pairwise voting agreement with every other UN member state.
codes = ideal_points.drop_duplicates().dropna()

d14 = (raw_d14.merge(codes.rename(columns={"ccode": "ccode1", "iso3c": "iso3_a"}), on="ccode1", how="left")
              .merge(codes.rename(columns={"ccode": "ccode2", "iso3c": "iso3_b"}), on="ccode2", how="left"))

# Each country appears on both sides of its dyads, so flatten before averaging.
flat = pd.concat([
    d14[["iso3_a", "year", "agree"]].rename(columns={"iso3_a": "market"}),
    d14[["iso3_b", "year", "agree"]].rename(columns={"iso3_b": "market"}),
], ignore_index=True)

annual = (flat[flat["market"].isin(MARKETS)]
          .groupby(["market", "year"], as_index=False)["agree"].mean())
annual["value"] = annual["agree"] * 100

df_d14 = to_contract(annual, "D14", labels="%")
df_d14.head(5)

In [ ]:
check_coverage(df_d14)

In [ ]:
load(df_d14)

## D15

In [ ]:
raw_d15 = wb.fetch("BM.KLT.DINV.CD.WD")

raw_d15.head(5)

In [ ]:
# Net-negative years (disinvestment) are dropped: this proxy is outward FDI
# scale, which cannot be negative.
df_d15 = to_contract(raw_d15[raw_d15["value"] > 0], "D15",
                     labels="USD (millions)", metric="MILLIONS", scale=1e6)
df_d15.head(5)

In [ ]:
check_coverage(df_d15)

In [ ]:
load(df_d15)

## D16

In [ ]:
# R&D coverage is far better as a share of GDP than as a level, so the level is
# reconstructed from the share and GDP.
raw_d16 = wb.fetch_many(["GB.XPD.RSDV.GD.ZS", "NY.GDP.MKTP.CD"])

raw_d16.head(5)

In [ ]:
d16 = raw_d16.dropna(subset=["GB.XPD.RSDV.GD.ZS", "NY.GDP.MKTP.CD"]).copy()
d16["value"] = d16["NY.GDP.MKTP.CD"] * d16["GB.XPD.RSDV.GD.ZS"] / 100

df_d16 = to_contract(d16[d16["value"] > 0], "D16",
                     labels="USD (millions)", metric="MILLIONS", scale=1e6, decimals=6)
df_d16.head(5)

In [ ]:
check_coverage(df_d16)

In [ ]:
load(df_d16)

# Topic 2: Technology

## D17

In [ ]:
raw_d17 = pd.read_csv(RAW_DIR / "Epoch AI" / "AI Capabilities" / "epoch_capabilities_index.csv")

raw_d17.head(5)

In [ ]:
# Annual global frontier: the year's best score, carried forward with a
# cumulative max so a frontier holds until something beats it.
d17 = raw_d17.copy()
d17["year"] = pd.to_datetime(d17["Release date"], errors="coerce").dt.year
d17["value"] = pd.to_numeric(d17["ECI Score"], errors="coerce")
d17 = d17.dropna(subset=["year", "value"])

frontier = d17.groupby("year", as_index=False)["value"].max().sort_values("year")
frontier["value"] = frontier["value"].cummax()
frontier["market"] = "GLO"

df_d17 = to_contract(frontier, "D17", labels="ECI")
df_d17.head(5)

In [ ]:
check_coverage(df_d17)

In [ ]:
load(df_d17)

## D18

In [ ]:
raw_d18 = pd.read_csv(RAW_DIR / "Epoch AI" / "AI Models" / "all_ai_models.csv",
                      low_memory=False)

raw_d18.head(5)

In [ ]:
# Keep only the models Epoch flags as frontier.
d18 = raw_d18[raw_d18["Frontier model"].astype(str).str.casefold()
              .isin({"true", "1", "yes"})].copy()
d18["year"] = pd.to_datetime(d18["Publication date"], errors="coerce").dt.year
d18["value"] = pd.to_numeric(d18["Training compute (FLOP)"], errors="coerce")
d18 = d18.dropna(subset=["year", "value"])

annual = d18.groupby("year", as_index=False)["value"].max().sort_values("year")
# A year with no new frontier model keeps the previous frontier: a state
# carry-forward, not interpolation between observations.
annual = (annual.set_index("year")
          .reindex(range(int(annual["year"].min()), int(annual["year"].max()) + 1))
          .rename_axis("year").reset_index())
annual["value"] = annual["value"].ffill().cummax()
annual["market"] = "GLO"

df_d18 = to_contract(annual, "D18", labels="FLOP", metric="QUADRILLION", scale=1e15)
df_d18.head(5)

In [ ]:
check_coverage(df_d18)

In [ ]:
load(df_d18)

## D19

In [ ]:
raw_d19 = pd.read_csv(RAW_DIR / "TOP500" / "top500_number_one.csv")

raw_d19.head(5)

In [ ]:
# TOP500 publishes twice a year; the year's frontier is the better of the two.
d19 = raw_d19.rename(columns={c: c.strip().lower() for c in raw_d19.columns})
annual = d19.groupby("year", as_index=False)["value"].max()
annual["market"] = "GLO"

df_d19 = to_contract(annual, "D19", labels="TFlop/s")
df_d19.head(5)

In [ ]:
check_coverage(df_d19)

In [ ]:
load(df_d19)

## D20

In [ ]:
raw_d20 = pd.read_excel(RAW_DIR / "NHGRI" / "sequencing_costs.xlsx",
                        sheet_name="Data Table")

raw_d20.head(5)

In [ ]:
# NHGRI samples several times a year; the last observation is the year-end cost.
d20 = raw_d20.copy()
d20["year"] = pd.to_datetime(d20["Date"], errors="coerce").dt.year
d20["value"] = pd.to_numeric(
    d20["Cost per Genome"].astype(str).str.replace(r"[$,]", "", regex=True),
    errors="coerce")
d20 = d20.dropna(subset=["year", "value"]).sort_values("Date")

annual = d20.groupby("year", as_index=False)["value"].last()
annual["market"] = "GLO"

df_d20 = to_contract(annual, "D20", labels="USD thousands per genome",
                     metric="THOUSANDS", scale=1e3)
df_d20.head(5)

In [ ]:
check_coverage(df_d20)

In [ ]:
load(df_d20)

## D22

In [ ]:
raw_d22 = wb.fetch("TX.VAL.TECH.CD")

raw_d22.head(5)

In [ ]:
df_d22 = to_contract(raw_d22, "D22", labels="USD (millions)",
                     metric="MILLIONS", scale=1e6, decimals=6)
df_d22.head(5)

In [ ]:
check_coverage(df_d22)

In [ ]:
load(df_d22)

## D23

In [ ]:
# ICT goods are published as shares of merchandise trade, so the balance needs
# export value and share plus import value and share.
raw_d23 = wb.fetch_many(["TX.VAL.MRCH.CD.WT", "TX.VAL.ICTG.ZS.UN",
                         "TM.VAL.MRCH.CD.WT", "TM.VAL.ICTG.ZS.UN"])

raw_d23.head(5)

In [ ]:
d23 = raw_d23.dropna().copy()
d23["value"] = (d23["TX.VAL.MRCH.CD.WT"] * d23["TX.VAL.ICTG.ZS.UN"] / 100
                - d23["TM.VAL.MRCH.CD.WT"] * d23["TM.VAL.ICTG.ZS.UN"] / 100)

# A trade balance is legitimately negative, so no positivity filter here.
df_d23 = to_contract(d23, "D23", labels="USD (millions)",
                     metric="MILLIONS", scale=1e6, decimals=6)
df_d23.head(5)

In [ ]:
check_coverage(df_d23)

In [ ]:
load(df_d23)

## D24

In [ ]:
raw_d24 = wb.fetch_many(["NV.IND.MANF.CD", "NV.MNF.TECH.ZS.UN"])

raw_d24.head(5)

In [ ]:
d24 = raw_d24.dropna().copy()
d24["value"] = d24["NV.IND.MANF.CD"] * d24["NV.MNF.TECH.ZS.UN"] / 100

df_d24 = to_contract(d24, "D24", labels="USD (millions)",
                     metric="MILLIONS", scale=1e6, decimals=6)
df_d24.head(5)

In [ ]:
check_coverage(df_d24)

In [ ]:
load(df_d24)

## D25

In [ ]:
raw_d25 = wb.fetch("SP.POP.SCIE.RD.P6")

raw_d25.head(5)

In [ ]:
df_d25 = to_contract(raw_d25, "D25", labels="No. of researchers per million people", decimals=6)
df_d25.head(5)

In [ ]:
check_coverage(df_d25)

In [ ]:
load(df_d25)

## D26

In [ ]:
raw_d26 = wb.fetch("IP.JRN.ARTC.SC")

raw_d26.head(5)

In [ ]:
df_d26 = to_contract(raw_d26, "D26", labels="No. of articles", decimals=6)
df_d26.head(5)

In [ ]:
check_coverage(df_d26)

In [ ]:
load(df_d26)

## D27

In [ ]:
raw_d27 = wb.fetch("BX.GSR.CCIS.CD")

raw_d27.head(5)

In [ ]:
df_d27 = to_contract(raw_d27, "D27", labels="USD (millions)",
                     metric="MILLIONS", scale=1e6, decimals=6)
df_d27.head(5)

In [ ]:
check_coverage(df_d27)

In [ ]:
load(df_d27)

## D28

In [ ]:
raw_d28 = wb.fetch("IT.NET.USER.ZS")

raw_d28.head(5)

In [ ]:
df_d28 = to_contract(raw_d28, "D28", labels="% of population", decimals=6)
df_d28.head(5)

In [ ]:
check_coverage(df_d28)

In [ ]:
load(df_d28)

## D29

In [ ]:
raw_d29 = wb.fetch("IT.NET.SECR.P6")

raw_d29.head(5)

In [ ]:
df_d29 = to_contract(raw_d29, "D29", labels="No. of servers per million people", decimals=6)
df_d29.head(5)

In [ ]:
check_coverage(df_d29)

In [ ]:
load(df_d29)

## D30

In [ ]:
raw_d30 = pd.read_csv(RAW_DIR / "Statista" / "data_center_market_revenue.csv",
                      low_memory=False)

raw_d30.head(5)

In [ ]:
# Total data-centre revenue = servers + storage + network infrastructure.
SEGMENTS_D30 = {"Servers", "Storage", "Network Infrastructure"}

d30 = raw_d30[raw_d30["Chart"].str.strip().eq("Revenue Comparison")
              & raw_d30["Market"].str.strip().isin(SEGMENTS_D30)].copy()
assert set(d30["Unit"].dropna().str.strip()) == {"million USD (US$)"}
assert set(d30["Market"].str.strip()) == SEGMENTS_D30

d30["market"] = to_iso3(d30["Name"])
long_d30 = melt_years(d30.dropna(subset=["market"]), ["market", "Market"])
long_d30["value"] = pd.to_numeric(
    long_d30["value"].astype(str).str.replace(",", "", regex=False), errors="coerce")
long_d30 = long_d30.dropna(subset=["value"])

totals_d30 = long_d30.groupby(["market", "year"], as_index=False).agg(
    value=("value", "sum"), segments=("Market", "nunique"))
# Drop country-years missing a component rather than reporting a partial total.
totals_d30 = totals_d30[totals_d30["segments"].eq(len(SEGMENTS_D30))]

df_d30 = to_contract(totals_d30, "D30", labels="USD (millions)", metric="MILLIONS")
df_d30.head(5)

In [ ]:
check_coverage(df_d30)

In [ ]:
load(df_d30)

## D31

In [ ]:
raw_d31 = pd.read_excel(RAW_DIR / "Statista" / "semiconductor_market_revenue.xlsx",
                        sheet_name="Global Comparison", header=None)

raw_d31.head(5)

In [ ]:
# Row 2 states the unit and row 3 holds the years; countries start at row 4.
unit_d31 = str(raw_d31.iloc[2, 0]).strip()
assert unit_d31 == "REVENUE COMPARISON in billion USD (US$)", unit_d31

years_d31 = pd.to_numeric(raw_d31.iloc[3, 1:], errors="coerce")
body_d31 = raw_d31.iloc[4:].copy()
body_d31.columns = ["country"] + list(years_d31)
body_d31 = body_d31[body_d31["country"].apply(lambda x: isinstance(x, str))]
body_d31["market"] = to_iso3(body_d31["country"])

d31 = melt_years(body_d31.dropna(subset=["market"]), "market")
d31["value"] = pd.to_numeric(d31["value"], errors="coerce") * 1_000.0  # bn -> mn USD

df_d31 = to_contract(d31, "D31", labels="USD (millions)", metric="MILLIONS")
df_d31.head(5)

In [ ]:
check_coverage(df_d31)

In [ ]:
load(df_d31)

## D32

In [ ]:
# Robot imports are HS 847950 throughout, plus HS 842870 from 2022.
robot_imports = pd.read_csv(RAW_DIR / "D32 Robot Intensity" / "comtrade_robot_imports.csv")
manufacturing_jobs = pd.read_csv(RAW_DIR / "D32 Robot Intensity" / "ilostat_manufacturing_employment.csv")
us_cpi = wb.fetch("FP.CPI.TOTL", markets=["USA"]).rename(columns={"value": "cpi"})

robot_imports.head(5)

In [ ]:
#   D32 = real_2015_USD(robot imports) / manufacturing employment * 10,000
BASE_YEAR = 2015

base_cpi = float(us_cpi.loc[us_cpi["year"] == BASE_YEAR, "cpi"].iloc[0])

d32 = (robot_imports.merge(manufacturing_jobs, on=["market", "year"], how="inner")
                    .merge(us_cpi[["year", "cpi"]], on="year", how="inner"))
d32["real_2015_usd"] = d32["imports_current_usd"] * base_cpi / d32["cpi"]
d32["value"] = d32["real_2015_usd"] / d32["employment_persons"] * 10_000.0
d32 = d32.replace([np.inf, -np.inf], np.nan).dropna(subset=["value"])

df_d32 = to_contract(d32[d32["value"] >= 0], "D32",
                     labels="2015 USD per 10,000 manufacturing workers", decimals=6)
df_d32.head(5)

In [ ]:
check_coverage(df_d32)

In [ ]:
load(df_d32)

## D33

In [5]:
raw_d33 = pd.read_csv(RAW_DIR / "ILOSTAT" / "employment_by_occupation_isco08_2digit.csv")

raw_d33.head(5)

,ref_area,source,indicator,sex,classif1,time,obs_value,obs_status,note_classif,note_indicator,note_source
0,ARE,BA:716,EMP_TEMP_SEX_OC2_NB,SEX_T,OC2_ISCO08_TOTAL,2022,7303.023,NaN,NaN,NaN,R1:3513
1,ARE,BA:716,EMP_TEMP_SEX_OC2_NB,SEX_T,OC2_ISCO08_25,2022,66.746,NaN,NaN,NaN,R1:3513
2,ARE,BA:716,EMP_TEMP_SEX_OC2_NB,SEX_T,OC2_ISCO08_35,2022,22.122,NaN,NaN,NaN,R1:3513
3,ARE,BA:716,EMP_TEMP_SEX_OC2_NB,SEX_T,OC2_ISCO08_TOTAL,2021,7165.128,NaN,NaN,NaN,R1:3513
4,ARE,BA:716,EMP_TEMP_SEX_OC2_NB,SEX_T,OC2_ISCO08_25,2021,78.105,NaN,NaN,NaN,R1:3513


In [6]:
# ICT specialists are ISCO-08 sub-major groups 25 (ICT professionals) and
# 35 (ICT technicians), as a share of total employment.
ICT_GROUPS = ["OC2_ISCO08_25", "OC2_ISCO08_35"]

specialists = (raw_d33[raw_d33["classif1"].isin(ICT_GROUPS)]
               .groupby(["ref_area", "time"], as_index=False)
               .agg(ict=("obs_value", "sum"), groups=("classif1", "nunique")))
employed = (raw_d33[raw_d33["classif1"].eq("OC2_ISCO08_TOTAL")]
            [["ref_area", "time", "obs_value"]].rename(columns={"obs_value": "total"}))

d33 = specialists.merge(employed, on=["ref_area", "time"])
# Drop country-years reporting only one of the two groups, so a partial
# numerator is never divided by the full denominator.
d33 = d33[d33["groups"].eq(len(ICT_GROUPS)) & d33["total"].gt(0)]
d33["value"] = d33["ict"] / d33["total"] * 100

df_d33 = to_contract(d33, "D33", labels="%",
                     market_col="ref_area", year_col="time")
df_d33.head(5)

,proxy_id,market,year,value,labels,metric
0,D33_ARE,ARE,2017,0.444693,%,None
1,D33_ARE,ARE,2018,1.116617,%,None
2,D33_ARE,ARE,2020,1.870601,%,None
3,D33_ARE,ARE,2021,1.463170,%,None
4,D33_ARE,ARE,2022,1.216866,%,None


In [7]:
check_coverage(df_d33)

=== Coverage [D33] ===
Total datapoints : 259   |   years: 2000-2025
Countries covered: 26 / 34
Points per country: min 1, max 23

Missing (8):
   CAN  Canada
   CHN  China
   JPN  Japan
   KAZ  Kazakhstan
   KOR  South Korea
   SAU  Saudi Arabia
   UKR  Ukraine
   ZAF  South Africa


,market,country,datapoints
0,USA,United States,23
1,ARG,Argentina,19
2,VNM,Vietnam,16
3,GBR,United Kingdom,15
4,POL,Poland,15
5,TUR,Turkey,15
6,BRA,Brazil,14
7,DEU,Germany,14
8,FRA,France,14
9,ITA,Italy,14


In [8]:
load(df_d33)

  Written: 259 rows


## D35

In [ ]:
raw_d35 = pd.read_csv(RAW_DIR / "UNESCO UIS" / "ict_graduates_share.csv",
                      low_memory=False)

raw_d35.head(5)

In [ ]:
assert set(raw_d35["INDICATOR_ID"].str.strip()) == {"FOSGP.5T8.F600"}

d35 = raw_d35.rename(columns={"COUNTRY_ID": "market", "YEAR": "year", "VALUE": "value"})

df_d35 = to_contract(d35[d35["market"].isin(MARKETS)], "D35", labels="% of graduates")
assert df_d35["value"].between(0, 100).all()
df_d35.head(5)

In [ ]:
check_coverage(df_d35)

In [ ]:
load(df_d35)

## D36

In [1]:
raw_d36 = pd.read_csv(RAW_DIR / "OECD" / "labour_productivity_growth.csv",
                      low_memory=False)

raw_d36.head(5)

NameError: name 'pd' is not defined

In [ ]:
# The export mixes several measures, so pin every dimension before reading
# values: GDPHRS = GDP per hour, GY = growth on the previous year.
DIMENSIONS_D36 = {
    "FREQ": "A", "MEASURE": "GDPHRS", "ACTIVITY": "_T", "UNIT_MEASURE": "XDC_H",
    "PRICE_BASE": "L", "TRANSFORMATION": "GY", "ASSET_CODE": "_Z",
    "CONVERSION_TYPE": "_Z",
}
for column, wanted in DIMENSIONS_D36.items():
    assert set(raw_d36[column].dropna().astype(str).str.strip()) == {wanted}, column

d36 = raw_d36.rename(columns={"REF_AREA": "market", "TIME_PERIOD": "year",
                              "OBS_VALUE": "value"})

df_d36 = to_contract(d36[d36["market"].isin(MARKETS)], "D36", labels="%", decimals=6)
df_d36.head(5)

In [ ]:
check_coverage(df_d36)

In [ ]:
load(df_d36)

## D39

In [ ]:
raw_d39 = pd.read_csv(RAW_DIR / "ILOSTAT" / "labour_income_share.csv", low_memory=False)

raw_d39.head(5)

In [ ]:
assert set(raw_d39["indicator"].str.strip()) == {"SDG_1041_NOC_RT"}

d39 = raw_d39.rename(columns={"ref_area": "market", "time": "year",
                              "obs_value": "value"})

df_d39 = to_contract(d39[d39["market"].isin(MARKETS)], "D39", labels="% of GDP")
df_d39.head(5)

In [ ]:
check_coverage(df_d39)

In [ ]:
load(df_d39)

## D42

In [ ]:
raw_d42 = pd.read_csv(RAW_DIR / "OECD" / "tiva_ict_domestic_value_added_share.csv",
                      low_memory=False)

raw_d42.head(5)

In [ ]:
# Domestic value added in ICT/electronics (ISIC Rev.4 division C26) exports to
# the World. OECD publishes the share directly under PT_EXGR, so no division.
d42 = raw_d42[
    raw_d42["MEASURE"].str.strip().eq("EXGR_DVA")
    & raw_d42["ACTIVITY"].str.strip().eq("C26")
    & raw_d42["COUNTERPART_AREA"].str.strip().eq("W")
    & raw_d42["UNIT_MEASURE"].str.strip().eq("PT_EXGR")
    & raw_d42["FREQ"].str.strip().eq("A")
].rename(columns={"REF_AREA": "market", "TIME_PERIOD": "year", "OBS_VALUE": "value"})
assert set(pd.to_numeric(d42["UNIT_MULT"]).dropna().astype(int)) == {0}

df_d42 = to_contract(d42[d42["market"].isin(MARKETS)], "D42", labels="%")
assert df_d42["value"].between(0, 100).all()
df_d42.head(5)

In [ ]:
check_coverage(df_d42)

In [ ]:
load(df_d42)

## D43

In [ ]:
raw_d43 = wb.fetch_many(["BX.GSR.ROYL.CD", "BM.GSR.ROYL.CD"])

raw_d43.head(5)

In [ ]:
# Net IP income = charges received for use of IP minus charges paid.
d43 = raw_d43.dropna().copy()
d43["value"] = d43["BX.GSR.ROYL.CD"] - d43["BM.GSR.ROYL.CD"]

df_d43 = to_contract(d43, "D43", labels="USD (millions)",
                     metric="MILLIONS", scale=1e6, decimals=6)
df_d43.head(5)

In [ ]:
check_coverage(df_d43)

In [ ]:
load(df_d43)

## D44

In [ ]:
raw_d44 = pd.read_csv(RAW_DIR / "WTO" / "digitally_delivered_services_trade.csv",
                      low_memory=False)

raw_d44.head(5)

In [ ]:
# Total digitally delivered services (DDS), partner World, Mode 1 supply.
d44 = raw_d44[
    raw_d44["INDICATOR"].str.strip().eq("DDS")
    & raw_d44["PARTNER"].str.strip().eq("World")
    & raw_d44["MODE"].str.strip().eq("Mode 1")
    & raw_d44["UNIT"].str.strip().eq("Million US dollar")
    & raw_d44["FLOW"].str.strip().isin({"X", "M"})
].copy()
d44["market"] = to_iso3(d44["REPORTER_NAME"])
d44["value"] = pd.to_numeric(d44["VALUE"], errors="coerce")
d44 = d44.dropna(subset=["market", "value"])

# Exports minus imports, so both flows must be present for a country-year.
balance = d44.pivot(index=["market", "YEAR"], columns="FLOW",
                    values="value").dropna().reset_index()
balance["value"] = balance["X"] - balance["M"]

df_d44 = to_contract(balance, "D44", labels="USD (millions)",
                     metric="MILLIONS", year_col="YEAR")
df_d44.head(5)

In [ ]:
check_coverage(df_d44)

In [ ]:
load(df_d44)

## D46

In [ ]:
# Published only split by sex, so the disaggregated series is kept.
raw_d46 = wb.fetch_many_by_sex(["IT.NET.USER.MA.ZS", "IT.NET.USER.FE.ZS"])

raw_d46.head(5)

In [ ]:
# Gender gap in percentage points: male use minus female use.
d46 = raw_d46.dropna().copy()
d46["value"] = d46["IT.NET.USER.MA.ZS"] - d46["IT.NET.USER.FE.ZS"]

df_d46 = to_contract(d46, "D46", labels="%", decimals=6)
df_d46.head(5)

In [ ]:
check_coverage(df_d46)

In [ ]:
load(df_d46)

## D47

In [ ]:
raw_d47 = pd.read_csv(RAW_DIR / "ITU" / "urban_rural_internet_use.csv", low_memory=False)

raw_d47.head(5)

In [ ]:
assert set(raw_d47["seriesUnits"].dropna().str.strip()) == {"%"}

d47 = raw_d47.rename(columns={"entityIso": "market", "dataYear": "year",
                              "dataValue": "value"})
d47["value"] = pd.to_numeric(d47["value"], errors="coerce")
d47 = d47[d47["market"].isin(MARKETS)].dropna(subset=["value"])

# Urban minus rural, in percentage points; both halves must be present.
gap_d47 = d47.pivot(index=["market", "year"], columns="location_group",
                    values="value")[["Urban", "Rural"]].dropna().reset_index()
gap_d47["value"] = gap_d47["Urban"] - gap_d47["Rural"]

df_d47 = to_contract(gap_d47, "D47", labels="%")
df_d47.head(5)

In [ ]:
check_coverage(df_d47)

In [ ]:
load(df_d47)

## D48

In [ ]:
raw_d48 = pd.read_excel(RAW_DIR / "ITU" / "mobile_broadband_affordability.xlsx",
                        sheet_name="economies_2008-2025")

raw_d48.head(5)

In [ ]:
# ITU redefined the reference basket several times; take the official basket
# for each year so no single year mixes baskets.
BASKET_BY_YEAR = {
    **{y: "i271md_pd_B1GB_GNI" for y in range(2013, 2018)},   # 1 GB prepaid
    **{y: "i271mb_1GB5_GNI" for y in range(2018, 2021)},      # 1.5 GB
    **{y: "i271mb_2GB_GNI" for y in range(2021, 2025)},       # 2 GB
    2025: "i271mb_5GB_GNI",                                   # 5 GB
}

rows_d48 = []
for year, code in BASKET_BY_YEAR.items():
    basket = raw_d48[raw_d48["Code"].str.strip().eq(code)
                     & raw_d48["Unit"].str.strip().eq("GNIpc")]
    rows_d48.append(pd.DataFrame({"market": basket["IsoCode"].str.upper(),
                                  "year": year,
                                  "value": pd.to_numeric(basket[year], errors="coerce")}))
d48 = pd.concat(rows_d48, ignore_index=True).dropna(subset=["value"])

df_d48 = to_contract(d48[d48["market"].isin(MARKETS)], "D48",
                     labels="% of GNI per capita")
df_d48.head(5)

In [ ]:
check_coverage(df_d48)

In [ ]:
load(df_d48)

## D49

In [ ]:
raw_d49 = vdem_country_year()

raw_d49.head(5)

In [ ]:
df_d49 = vdem_indicator(raw_d49, "D49", "v2smgovsmmon_osp")
df_d49.head(5)

In [ ]:
check_coverage(df_d49)

In [ ]:
load(df_d49)

## D50

In [ ]:
raw_d50 = vdem_country_year()

raw_d50.head(5)

In [ ]:
df_d50 = vdem_indicator(raw_d50, "D50", "v2smgovfilprc_osp")
df_d50.head(5)

In [ ]:
check_coverage(df_d50)

In [ ]:
load(df_d50)

## D51

In [ ]:
raw_d51 = vdem_country_year()

raw_d51.head(5)

In [ ]:
df_d51 = vdem_indicator(raw_d51, "D51", "v2smgovdom_osp")
df_d51.head(5)

In [ ]:
check_coverage(df_d51)

In [ ]:
load(df_d51)

## D52

In [ ]:
raw_d52 = vdem_country_year()

raw_d52.head(5)

In [ ]:
df_d52 = vdem_indicator(raw_d52, "D52", "v2smorgavgact_osp")
df_d52.head(5)

In [ ]:
check_coverage(df_d52)

In [ ]:
load(df_d52)

## D53

In [ ]:
raw_d53 = vdem_country_year()

raw_d53.head(5)

In [ ]:
df_d53 = vdem_indicator(raw_d53, "D53", "v2smarrest_osp")
df_d53.head(5)

In [ ]:
check_coverage(df_d53)

In [ ]:
load(df_d53)

## D54

In [ ]:
raw_d54 = vdem_country_year()

raw_d54.head(5)

In [ ]:
df_d54 = vdem_indicator(raw_d54, "D54", "v2smprivcon_osp")
df_d54.head(5)

In [ ]:
check_coverage(df_d54)

In [ ]:
load(df_d54)

## D55

In [ ]:
raw_d55 = pd.read_csv(RAW_DIR / "Freedom House" / "freedom_on_the_net.csv")

raw_d55.head(5)

In [ ]:
d55 = raw_d55.copy()
d55["market"] = to_iso3(d55["country_slug"].str.replace("-", " "))
d55 = d55.dropna(subset=["market"])

df_d55 = to_contract(d55, "D55", labels="Index")
assert df_d55["value"].between(0, 100).all()
df_d55.head(5)

In [ ]:
check_coverage(df_d55)

In [ ]:
load(df_d55)

## D56

In [ ]:
raw_d56 = pd.read_csv(RAW_DIR / "UN DESA" / "e_participation_index.csv")

raw_d56.head(5)

In [ ]:
d56 = raw_d56.copy()
d56["market"] = to_iso3(d56["country"])
d56 = d56.dropna(subset=["market"])

df_d56 = to_contract(d56, "D56", labels="Index")
assert df_d56["value"].between(0, 1).all()
df_d56.head(5)

In [ ]:
check_coverage(df_d56)

In [ ]:
load(df_d56)

# Topic 3: Planet

Topics 3 to 5 read from `Raw Data/Topic 3-5/Prepared/`. `read_prepared()` drops
the publisher's projection and scenario rows so only observations reach the sheet.

## D58

In [ ]:
raw_d58 = read_prepared("D58")

raw_d58.head(5)

In [ ]:
df_d58 = to_contract(raw_d58, "D58", labels="\u00b0C")
df_d58.head(5)

In [ ]:
check_coverage(df_d58)

In [ ]:
load(df_d58)

## D59

In [ ]:
raw_d59 = read_prepared("D59")

raw_d59.head(5)

In [ ]:
df_d59 = to_contract(raw_d59, "D59", labels="ppm")
df_d59.head(5)

In [ ]:
check_coverage(df_d59)

In [ ]:
load(df_d59)

## D60

In [ ]:
raw_d60 = read_prepared("D60")

raw_d60.head(5)

In [ ]:
df_d60 = to_contract(raw_d60, "D60", labels="mm")
df_d60.head(5)

In [ ]:
check_coverage(df_d60)

In [ ]:
load(df_d60)

## D61

In [ ]:
raw_d61 = read_prepared("D61")

raw_d61.head(5)

In [ ]:
df_d61 = to_contract(raw_d61, "D61", labels="10^22 Joules")
df_d61.head(5)

In [ ]:
check_coverage(df_d61)

In [ ]:
load(df_d61)

## D61a

In [ ]:
raw_d61a = read_prepared("D61a")

raw_d61a.head(5)

In [ ]:
df_d61a = to_contract(raw_d61a, "D61a", labels="pH")
df_d61a.head(5)

In [ ]:
check_coverage(df_d61a)

In [ ]:
load(df_d61a)

## D62

In [ ]:
raw_d62 = read_prepared("D62")

raw_d62.head(5)

In [ ]:
df_d62 = to_contract(raw_d62, "D62", labels="Tonnes of CO2 equivalent (millions)", metric="MILLIONS")
df_d62.head(5)

In [ ]:
check_coverage(df_d62)

In [ ]:
load(df_d62)

## D63

In [ ]:
raw_d63 = read_prepared("D63")

raw_d63.head(5)

In [ ]:
df_d63 = to_contract(raw_d63, "D63", labels="Tonnes of CO2 equivalent per capita")
df_d63.head(5)

In [ ]:
check_coverage(df_d63)

In [ ]:
load(df_d63)

## D64

In [ ]:
raw_d64 = read_prepared("D64")

raw_d64.head(5)

In [ ]:
df_d64 = to_contract(raw_d64, "D64", labels="Tonnes of CO2 equivalent per capita")
df_d64.head(5)

In [ ]:
check_coverage(df_d64)

In [ ]:
load(df_d64)

## D65

In [ ]:
raw_d65 = read_prepared("D65")

raw_d65.head(5)

In [ ]:
df_d65 = to_contract(raw_d65, "D65", labels="\u00b0C")
df_d65.head(5)

In [ ]:
check_coverage(df_d65)

In [ ]:
load(df_d65)

## D66

In [ ]:
raw_d66 = read_prepared("D66")

raw_d66.head(5)

In [ ]:
df_d66 = to_contract(raw_d66, "D66", labels="days")
df_d66.head(5)

In [ ]:
check_coverage(df_d66)

In [ ]:
load(df_d66)

## D67

In [ ]:
raw_d67 = read_prepared("D67")

raw_d67.head(5)

In [ ]:
df_d67 = to_contract(raw_d67, "D67", labels="mm")
df_d67.head(5)

In [ ]:
check_coverage(df_d67)

In [ ]:
load(df_d67)

## D68

In [ ]:
raw_d68 = read_prepared("D68")

raw_d68.head(5)

In [ ]:
df_d68 = to_contract(raw_d68, "D68", labels="days")
df_d68.head(5)

In [ ]:
check_coverage(df_d68)

In [ ]:
load(df_d68)

## D70

In [ ]:
raw_d70 = read_prepared("D70")

raw_d70.head(5)

In [ ]:
df_d70 = to_contract(raw_d70, "D70", labels="\u00b5g/m\u00b3")
df_d70.head(5)

In [ ]:
check_coverage(df_d70)

In [ ]:
load(df_d70)

## D71

In [ ]:
raw_d71 = read_prepared("D71")

raw_d71.head(5)

In [ ]:
df_d71 = to_contract(raw_d71, "D71", labels="No. of deaths per 100,000 people")
df_d71.head(5)

In [ ]:
check_coverage(df_d71)

In [ ]:
load(df_d71)

## D72

In [ ]:
raw_d72 = read_prepared("D72")

raw_d72.head(5)

In [ ]:
df_d72 = to_contract(raw_d72, "D72", labels="No. of people per 100,000 people")
df_d72.head(5)

In [ ]:
check_coverage(df_d72)

In [ ]:
load(df_d72)

## D73

In [ ]:
raw_d73 = read_prepared("D73")

raw_d73.head(5)

In [ ]:
df_d73 = to_contract(raw_d73, "D73", labels="%")
df_d73.head(5)

In [ ]:
check_coverage(df_d73)

In [ ]:
load(df_d73)

## D74

In [ ]:
raw_d74 = read_prepared("D74")

raw_d74.head(5)

In [ ]:
df_d74 = to_contract(raw_d74, "D74", labels="Cubic metres per capita")
df_d74.head(5)

In [ ]:
check_coverage(df_d74)

In [ ]:
load(df_d74)

## D75

In [ ]:
raw_d75 = read_prepared("D75")

raw_d75.head(5)

In [ ]:
df_d75 = to_contract(raw_d75, "D75", labels="USD per cubic metre")
df_d75.head(5)

In [ ]:
check_coverage(df_d75)

In [ ]:
load(df_d75)

## D76

In [ ]:
raw_d76 = read_prepared("D76")

raw_d76.head(5)

In [ ]:
df_d76 = to_contract(raw_d76, "D76", labels="kg per hectare")
df_d76.head(5)

In [ ]:
check_coverage(df_d76)

In [ ]:
load(df_d76)

## D77

In [ ]:
raw_d77 = read_prepared("D77")

raw_d77.head(5)

In [ ]:
df_d77 = to_contract(raw_d77, "D77", labels="Index")
df_d77.head(5)

In [ ]:
check_coverage(df_d77)

In [ ]:
load(df_d77)

## D78

In [ ]:
raw_d78 = read_prepared("D78")

raw_d78.head(5)

In [ ]:
df_d78 = to_contract(raw_d78, "D78", labels="%")
df_d78.head(5)

In [ ]:
check_coverage(df_d78)

In [ ]:
load(df_d78)

## D79

In [ ]:
raw_d79 = read_prepared("D79")

raw_d79.head(5)

In [ ]:
df_d79 = to_contract(raw_d79, "D79", labels="Index")
df_d79.head(5)

In [ ]:
check_coverage(df_d79)

In [ ]:
load(df_d79)

## D82

In [ ]:
raw_d82 = read_prepared("D82")

raw_d82.head(5)

In [ ]:
df_d82 = to_contract(raw_d82, "D82", labels="Index")
df_d82.head(5)

In [ ]:
check_coverage(df_d82)

In [ ]:
load(df_d82)

## D83

In [ ]:
raw_d83 = read_prepared("D83")

raw_d83.head(5)

In [ ]:
df_d83 = to_contract(raw_d83, "D83", labels="%")
df_d83.head(5)

In [ ]:
check_coverage(df_d83)

In [ ]:
load(df_d83)

## D84

In [ ]:
raw_d84 = read_prepared("D84")

raw_d84.head(5)

In [ ]:
df_d84 = to_contract(raw_d84, "D84", labels="Watt per capita")
df_d84.head(5)

In [ ]:
check_coverage(df_d84)

In [ ]:
load(df_d84)

## D85

In [ ]:
raw_d85 = read_prepared("D85")

raw_d85.head(5)

In [ ]:
df_d85 = to_contract(raw_d85, "D85", labels="%")
df_d85.head(5)

In [ ]:
check_coverage(df_d85)

In [ ]:
load(df_d85)

## D86

In [ ]:
raw_d86 = read_prepared("D86")

raw_d86.head(5)

In [ ]:
df_d86 = to_contract(raw_d86, "D86", labels="MJ per USD")
df_d86.head(5)

In [ ]:
check_coverage(df_d86)

In [ ]:
load(df_d86)

## D87

In [ ]:
raw_d87 = read_prepared("D87")

raw_d87.head(5)

In [ ]:
df_d87 = to_contract(raw_d87, "D87", labels="gCO2 per kWh")
df_d87.head(5)

In [ ]:
check_coverage(df_d87)

In [ ]:
load(df_d87)

## D88

In [ ]:
raw_d88 = read_prepared("D88")

raw_d88.head(5)

In [ ]:
df_d88 = to_contract(raw_d88, "D88", labels="%")
df_d88.head(5)

In [ ]:
check_coverage(df_d88)

In [ ]:
load(df_d88)

## D89

In [ ]:
raw_d89 = read_prepared("D89")

raw_d89.head(5)

In [ ]:
df_d89 = to_contract(raw_d89, "D89", labels="%")
df_d89.head(5)

In [ ]:
check_coverage(df_d89)

In [ ]:
load(df_d89)

## D90

In [ ]:
raw_d90 = pd.read_csv(RAW_DIR / "Topic 3-5" / "UNEP IRP" / "Original"
                      / "mfa_totals_ratios_export.csv")

raw_d90.head(5)

In [ ]:
# The export carries many flow types in one file, so pin the flow before melting.
d90 = raw_d90[
    raw_d90["Flow code"].eq("MF/cap")
    & raw_d90["Flow name"].eq("Material Footprint (RMC) per capita")
    & raw_d90["Flow unit"].eq("t/cap")
].copy()
d90["market"] = to_iso3(d90["Country"])
d90 = d90.dropna(subset=["market"])
assert not d90["market"].duplicated().any()

long_d90 = melt_years(d90, "market")

df_d90 = to_contract(long_d90, "D90", labels="Tonnes per capita")
df_d90.head(5)

In [ ]:
check_coverage(df_d90)

In [ ]:
load(df_d90)

## D91

In [ ]:
# Exports come from three bulk downloads; imports from the per-year API pulls.
D91_CODES = {"250410", "253090", "260300", "260400", "260500", "283691"}
D91_DIR = RAW_DIR / "Topic 3-5" / "UN Comtrade" / "Original"

exports_d91 = pd.concat([
    pd.read_csv(D91_DIR / name, encoding="cp1252", index_col=False, low_memory=False)
    for name in ["TradeData_8_6_2026_23_33_39.csv",
                 "TradeData_8_6_2026_23_35_9.csv",
                 "TradeData_8_6_2026_23_35_44.csv"]
], ignore_index=True)

imports_d91 = pd.concat([
    pd.json_normalize(json.loads((D91_DIR / "API Imports" / f"D91_imports_{year}.json")
                                 .read_text())["data"])
    for year in range(START_YEAR, END_YEAR + 1)
], ignore_index=True)

exports_d91.head(5)

In [ ]:
# The import pulls carry only the numeric reporterCode, so take the code -> ISO3
# map from the exports.
reporter_iso = dict(exports_d91.loc[exports_d91["reporterISO"].isin(MARKETS),
                                    ["reporterCode", "reporterISO"]]
                    .drop_duplicates().itertuples(index=False))
assert set(reporter_iso.values()) == set(MARKETS)

exports_d91["market"] = exports_d91["reporterISO"]
imports_d91["market"] = imports_d91["reporterCode"].map(reporter_iso)

trade_d91 = pd.concat([exports_d91, imports_d91], ignore_index=True)
trade_d91["cmdCode"] = trade_d91["cmdCode"].astype(str).str.zfill(6)
trade_d91 = trade_d91[trade_d91["market"].isin(MARKETS)
                      & trade_d91["cmdCode"].isin(D91_CODES)]
assert set(trade_d91["cmdCode"]) == D91_CODES
assert trade_d91["primaryValue"].ge(0).all()

# Net import dependence = 100 * (imports - exports) / imports over the basket,
# floored at 0 so a net exporter scores 0 rather than a negative number.
flows_d91 = (trade_d91.groupby(["market", "refYear", "flowCode"], as_index=False)
             ["primaryValue"].sum())
wide_d91 = (flows_d91.pivot(index=["market", "refYear"], columns="flowCode",
                            values="primaryValue")
            .rename_axis(columns=None).reset_index())
wide_d91[["M", "X"]] = wide_d91.reindex(columns=["M", "X"]).fillna(0.0)
wide_d91 = wide_d91[(wide_d91["M"] > 0) | (wide_d91["X"] > 0)]
wide_d91["value"] = np.where(
    wide_d91["M"] > 0,
    100.0 * (wide_d91["M"] - wide_d91["X"]).clip(lower=0) / wide_d91["M"],
    0.0)

df_d91 = to_contract(wide_d91, "D91", labels="%", year_col="refYear")
df_d91.head(5)

In [ ]:
check_coverage(df_d91)

In [ ]:
load(df_d91)

## D92

In [ ]:
raw_d92 = read_prepared("D92")

raw_d92.head(5)

In [ ]:
df_d92 = to_contract(raw_d92, "D92", labels="Index")
df_d92.head(5)

In [ ]:
check_coverage(df_d92)

In [ ]:
load(df_d92)

## D93

In [ ]:
raw_d93 = read_prepared("D93")

raw_d93.head(5)

In [ ]:
df_d93 = to_contract(raw_d93, "D93", labels="Index")
df_d93.head(5)

In [ ]:
check_coverage(df_d93)

In [ ]:
load(df_d93)

## D94

In [ ]:
raw_d94 = read_prepared("D94")

raw_d94.head(5)

In [ ]:
df_d94 = to_contract(raw_d94, "D94", labels="Index")
df_d94.head(5)

In [ ]:
check_coverage(df_d94)

In [ ]:
load(df_d94)

## D95

In [ ]:
raw_d95 = read_prepared("D95")

raw_d95.head(5)

In [ ]:
df_d95 = to_contract(raw_d95, "D95", labels="% of GDP")
df_d95.head(5)

In [ ]:
check_coverage(df_d95)

In [ ]:
load(df_d95)

## D97

In [ ]:
raw_d97 = read_prepared("D97")

raw_d97.head(5)

In [ ]:
df_d97 = to_contract(raw_d97, "D97", labels="Index")
df_d97.head(5)

In [ ]:
check_coverage(df_d97)

In [ ]:
load(df_d97)

## D98

In [ ]:
raw_d98 = read_prepared("D98")

raw_d98.head(5)

In [ ]:
df_d98 = to_contract(raw_d98, "D98", labels="USD per capita")
df_d98.head(5)

In [ ]:
check_coverage(df_d98)

In [ ]:
load(df_d98)

# Topic 4: People

## D100

In [ ]:
raw_d100 = read_prepared("D100")

raw_d100.head(5)

In [ ]:
df_d100 = to_contract(raw_d100, "D100", labels="%")
df_d100.head(5)

In [ ]:
check_coverage(df_d100)

In [ ]:
load(df_d100)

## D101

In [ ]:
raw_d101 = read_prepared("D101")

raw_d101.head(5)

In [ ]:
df_d101 = to_contract(raw_d101, "D101", labels="No. of births per woman")
df_d101.head(5)

In [ ]:
check_coverage(df_d101)

In [ ]:
load(df_d101)

## D102

In [ ]:
raw_d102 = read_prepared("D102")

raw_d102.head(5)

In [ ]:
df_d102 = to_contract(raw_d102, "D102", labels="Years")
df_d102.head(5)

In [ ]:
check_coverage(df_d102)

In [ ]:
load(df_d102)

## D103

In [ ]:
raw_d103 = read_prepared("D103")

raw_d103.head(5)

In [ ]:
df_d103 = to_contract(raw_d103, "D103", labels="Years")
df_d103.head(5)

In [ ]:
check_coverage(df_d103)

In [ ]:
load(df_d103)

## D104

In [ ]:
raw_d104 = read_prepared("D104")

raw_d104.head(5)

In [ ]:
df_d104 = to_contract(raw_d104, "D104", labels="%")
df_d104.head(5)

In [ ]:
check_coverage(df_d104)

In [ ]:
load(df_d104)

## D105

In [ ]:
raw_d105 = read_prepared("D105")

raw_d105.head(5)

In [ ]:
df_d105 = to_contract(raw_d105, "D105", labels="%")
df_d105.head(5)

In [ ]:
check_coverage(df_d105)

In [ ]:
load(df_d105)

## D106

In [ ]:
raw_d106 = read_prepared("D106")

raw_d106.head(5)

In [ ]:
df_d106 = to_contract(raw_d106, "D106", labels="No, of people per 100 working-age people")
df_d106.head(5)

In [ ]:
check_coverage(df_d106)

In [ ]:
load(df_d106)

## D107

In [ ]:
raw_d107 = read_prepared("D107")

raw_d107.head(5)

In [ ]:
df_d107 = to_contract(raw_d107, "D107", labels="%")
df_d107.head(5)

In [ ]:
check_coverage(df_d107)

In [ ]:
load(df_d107)

## D108

In [ ]:
raw_d108 = read_prepared("D108")

raw_d108.head(5)

In [ ]:
df_d108 = to_contract(raw_d108, "D108", labels="%")
df_d108.head(5)

In [ ]:
check_coverage(df_d108)

In [ ]:
load(df_d108)

## D109

In [ ]:
raw_d109 = read_prepared("D109")

raw_d109.head(5)

In [ ]:
df_d109 = to_contract(raw_d109, "D109", labels="%")
df_d109.head(5)

In [ ]:
check_coverage(df_d109)

In [ ]:
load(df_d109)

## D110

In [ ]:
raw_d110 = read_prepared("D110")

raw_d110.head(5)

In [ ]:
df_d110 = to_contract(raw_d110, "D110", labels="%")
df_d110.head(5)

In [ ]:
check_coverage(df_d110)

In [ ]:
load(df_d110)

## D111

In [ ]:
raw_d111 = read_prepared("D111")

raw_d111.head(5)

In [ ]:
df_d111 = to_contract(raw_d111, "D111", labels="%")
df_d111.head(5)

In [ ]:
check_coverage(df_d111)

In [ ]:
load(df_d111)

## D112

In [ ]:
raw_d112 = read_prepared("D112")

raw_d112.head(5)

In [ ]:
df_d112 = to_contract(raw_d112, "D112", labels="No. of migrants per 1,000 people")
df_d112.head(5)

In [ ]:
check_coverage(df_d112)

In [ ]:
load(df_d112)

## D113

In [ ]:
raw_d113 = read_prepared("D113")

raw_d113.head(5)

In [ ]:
df_d113 = to_contract(raw_d113, "D113", labels="No. of refugees per 100,000 people")
df_d113.head(5)

In [ ]:
check_coverage(df_d113)

In [ ]:
load(df_d113)

## D114

In [ ]:
raw_d114 = read_prepared("D114")

raw_d114.head(5)

In [ ]:
df_d114 = to_contract(raw_d114, "D114", labels="No. of displacement per 100,000 people")
df_d114.head(5)

In [ ]:
check_coverage(df_d114)

In [ ]:
load(df_d114)

## D115

In [ ]:
raw_d115 = read_prepared("D115")

raw_d115.head(5)

In [ ]:
df_d115 = to_contract(raw_d115, "D115", labels="% of GDP")
df_d115.head(5)

In [ ]:
check_coverage(df_d115)

In [ ]:
load(df_d115)

## D116

In [ ]:
raw_d116 = read_prepared("D116")

raw_d116.head(5)

In [ ]:
df_d116 = to_contract(raw_d116, "D116", labels="Years")
df_d116.head(5)

In [ ]:
check_coverage(df_d116)

In [ ]:
load(df_d116)

## D117

In [ ]:
raw_d117 = read_prepared("D117")

raw_d117.head(5)

In [ ]:
df_d117 = to_contract(raw_d117, "D117", labels="No. of women per 100,000 live births")
df_d117.head(5)

In [ ]:
check_coverage(df_d117)

In [ ]:
load(df_d117)

## D118

In [ ]:
raw_d118 = read_prepared("D118")

raw_d118.head(5)

In [ ]:
df_d118 = to_contract(raw_d118, "D118", labels="No. of people per 1,000 live births")
df_d118.head(5)

In [ ]:
check_coverage(df_d118)

In [ ]:
load(df_d118)

## D119

In [ ]:
raw_d119 = read_prepared("D119")

raw_d119.head(5)

In [ ]:
df_d119 = to_contract(raw_d119, "D119", labels="Index")
df_d119.head(5)

In [ ]:
check_coverage(df_d119)

In [ ]:
load(df_d119)

## D120

In [ ]:
raw_d120 = read_prepared("D120")

raw_d120.head(5)

In [ ]:
df_d120 = to_contract(raw_d120, "D120", labels="USD per capita")
df_d120.head(5)

In [ ]:
check_coverage(df_d120)

In [ ]:
load(df_d120)

## D121

In [ ]:
raw_d121 = read_prepared("D121")

raw_d121.head(5)

In [ ]:
df_d121 = to_contract(raw_d121, "D121", labels="%")
df_d121.head(5)

In [ ]:
check_coverage(df_d121)

In [ ]:
load(df_d121)

## D123

In [ ]:
raw_d123 = read_prepared("D123")

raw_d123.head(5)

In [ ]:
df_d123 = to_contract(raw_d123, "D123", labels="%")
df_d123.head(5)

In [ ]:
check_coverage(df_d123)

In [ ]:
load(df_d123)

## D124

In [ ]:
raw_d124 = read_prepared("D124")

raw_d124.head(5)

In [ ]:
df_d124 = to_contract(raw_d124, "D124", labels="No. of nurses and midwives per 1,000 people")
df_d124.head(5)

In [ ]:
check_coverage(df_d124)

In [ ]:
load(df_d124)

## D124a

In [ ]:
raw_d124a = read_prepared("D124a")

raw_d124a.head(5)

In [ ]:
df_d124a = to_contract(raw_d124a, "D124a", labels="%")
df_d124a.head(5)

In [ ]:
check_coverage(df_d124a)

In [ ]:
load(df_d124a)

## D125

In [ ]:
raw_d125 = read_prepared("D125")

raw_d125.head(5)

In [ ]:
df_d125 = to_contract(raw_d125, "D125", labels="%")
df_d125.head(5)

In [ ]:
check_coverage(df_d125)

In [ ]:
load(df_d125)

## D126

In [ ]:
raw_d126 = read_prepared("D126")

raw_d126.head(5)

In [ ]:
df_d126 = to_contract(raw_d126, "D126", labels="Index")
df_d126.head(5)

In [ ]:
check_coverage(df_d126)

In [ ]:
load(df_d126)

## D127

In [ ]:
raw_d127 = read_prepared("D127")

raw_d127.head(5)

In [ ]:
df_d127 = to_contract(raw_d127, "D127", labels="%")
df_d127.head(5)

In [ ]:
check_coverage(df_d127)

In [ ]:
load(df_d127)

## D128

In [ ]:
raw_d128 = read_prepared("D128")

raw_d128.head(5)

In [ ]:
df_d128 = to_contract(raw_d128, "D128", labels="%")
df_d128.head(5)

In [ ]:
check_coverage(df_d128)

In [ ]:
load(df_d128)

## D129

In [ ]:
raw_d129 = read_prepared("D129")

raw_d129.head(5)

In [ ]:
df_d129 = to_contract(raw_d129, "D129", labels="%")
df_d129.head(5)

In [ ]:
check_coverage(df_d129)

In [ ]:
load(df_d129)

## D131

In [ ]:
raw_d131 = read_prepared("D131")

raw_d131.head(5)

In [ ]:
df_d131 = to_contract(raw_d131, "D131", labels="%")
df_d131.head(5)

In [ ]:
check_coverage(df_d131)

In [ ]:
load(df_d131)

## D132

In [ ]:
raw_d132 = read_prepared("D132")

raw_d132.head(5)

In [ ]:
df_d132 = to_contract(raw_d132, "D132", labels="%")
df_d132.head(5)

In [ ]:
check_coverage(df_d132)

In [ ]:
load(df_d132)

## D133

In [ ]:
raw_d133 = read_prepared("D133")

raw_d133.head(5)

In [ ]:
df_d133 = to_contract(raw_d133, "D133", labels="%")
df_d133.head(5)

In [ ]:
check_coverage(df_d133)

In [ ]:
load(df_d133)

## D133a

In [ ]:
raw_d133a = read_prepared("D133a")

raw_d133a.head(5)

In [ ]:
df_d133a = to_contract(raw_d133a, "D133a", labels="%")
df_d133a.head(5)

In [ ]:
check_coverage(df_d133a)

In [ ]:
load(df_d133a)

## D134

In [ ]:
raw_d134 = read_prepared("D134")

raw_d134.head(5)

In [ ]:
df_d134 = to_contract(raw_d134, "D134", labels="Index")
df_d134.head(5)

In [ ]:
check_coverage(df_d134)

In [ ]:
load(df_d134)

## D135

In [ ]:
raw_d135 = read_prepared("D135")

raw_d135.head(5)

In [ ]:
df_d135 = to_contract(raw_d135, "D135", labels="Index")
df_d135.head(5)

In [ ]:
check_coverage(df_d135)

In [ ]:
load(df_d135)

## D136

In [ ]:
raw_d136 = read_prepared("D136")

raw_d136.head(5)

In [ ]:
df_d136 = to_contract(raw_d136, "D136", labels="Index")
df_d136.head(5)

In [ ]:
check_coverage(df_d136)

In [ ]:
load(df_d136)

## D137

In [ ]:
raw_d137 = read_prepared("D137")

raw_d137.head(5)

In [ ]:
df_d137 = to_contract(raw_d137, "D137", labels="No. of demonstration events (per 1,000,000 people)")
df_d137.head(5)

In [ ]:
check_coverage(df_d137)

In [ ]:
load(df_d137)

## D138

In [ ]:
raw_d138 = read_prepared("D138")

raw_d138.head(5)

In [ ]:
df_d138 = to_contract(raw_d138, "D138", labels="No. of political-violence fatalities (per 1,000,000 people)")
df_d138.head(5)

In [ ]:
check_coverage(df_d138)

In [ ]:
load(df_d138)

## D139

In [ ]:
raw_d139 = read_prepared("D139")

raw_d139.head(5)

In [ ]:
df_d139 = to_contract(raw_d139, "D139", labels="No.of intentional homicides (per 100,000 people)")
df_d139.head(5)

In [ ]:
check_coverage(df_d139)

In [ ]:
load(df_d139)

## D141

In [ ]:
raw_d141 = read_prepared("D141")

raw_d141.head(5)

In [ ]:
df_d141 = to_contract(raw_d141, "D141", labels="Index")
df_d141.head(5)

In [ ]:
check_coverage(df_d141)

In [ ]:
load(df_d141)

# Topic 5: Economy

## D142

In [ ]:
raw_d142 = read_prepared("D142")

raw_d142.head(5)

In [ ]:
df_d142 = to_contract(raw_d142, "D142", labels="USD (trillions)",
                     metric="TRILLIONS", scale=1e12)
df_d142.head(5)

In [ ]:
check_coverage(df_d142)

In [ ]:
load(df_d142)

## D143

In [ ]:
raw_d143 = read_prepared("D143")

raw_d143.head(5)

In [ ]:
df_d143 = to_contract(raw_d143, "D143", labels="%")
df_d143.head(5)

In [ ]:
check_coverage(df_d143)

In [ ]:
load(df_d143)

## D144

In [ ]:
raw_d144 = read_prepared("D144")

raw_d144.head(5)

In [ ]:
df_d144 = to_contract(raw_d144, "D144", labels="USD per capita (thousands)",
                     metric="THOUSAND", scale=1e3)
df_d144.head(5)

In [ ]:
check_coverage(df_d144)

In [ ]:
load(df_d144)

## D145

In [ ]:
raw_d145 = read_prepared("D145")

raw_d145.head(5)

In [ ]:
df_d145 = to_contract(raw_d145, "D145", labels="Constant 2021 PPP USD per capita (thousands)",
                     metric="THOUSAND", scale=1e3)
df_d145.head(5)

In [ ]:
check_coverage(df_d145)

In [ ]:
load(df_d145)

## D146

In [ ]:
raw_d146 = read_prepared("D146")

raw_d146.head(5)

In [ ]:
df_d146 = to_contract(raw_d146, "D146", labels="Index")
df_d146.head(5)

In [ ]:
check_coverage(df_d146)

In [ ]:
load(df_d146)

## D147

In [ ]:
raw_d147 = read_prepared("D147")

raw_d147.head(5)

In [ ]:
df_d147 = to_contract(raw_d147, "D147", labels="% of GDP")
df_d147.head(5)

In [ ]:
check_coverage(df_d147)

In [ ]:
load(df_d147)

## D148

In [ ]:
raw_d148 = read_prepared("D148")

raw_d148.head(5)

In [ ]:
df_d148 = to_contract(raw_d148, "D148", labels="% of GDP")
df_d148.head(5)

In [ ]:
check_coverage(df_d148)

In [ ]:
load(df_d148)

## D149

In [ ]:
raw_d149 = read_prepared("D149")

raw_d149.head(5)

In [ ]:
df_d149 = to_contract(raw_d149, "D149", labels="% of GDP")
df_d149.head(5)

In [ ]:
check_coverage(df_d149)

In [ ]:
load(df_d149)

## D150

In [ ]:
raw_d150 = read_prepared("D150")

raw_d150.head(5)

In [ ]:
df_d150 = to_contract(raw_d150, "D150", labels="% of GDP")
df_d150.head(5)

In [ ]:
check_coverage(df_d150)

In [ ]:
load(df_d150)

## D151

In [ ]:
raw_d151 = read_prepared("D151")

raw_d151.head(5)

In [ ]:
df_d151 = to_contract(raw_d151, "D151", labels="%")
df_d151.head(5)

In [ ]:
check_coverage(df_d151)

In [ ]:
load(df_d151)

## D152

In [ ]:
raw_d152 = read_prepared("D152")

raw_d152.head(5)

In [ ]:
df_d152 = to_contract(raw_d152, "D152", labels="%")
df_d152.head(5)

In [ ]:
check_coverage(df_d152)

In [ ]:
load(df_d152)

## D153

In [ ]:
raw_d153 = read_prepared("D153")

raw_d153.head(5)

In [ ]:
df_d153 = to_contract(raw_d153, "D153", labels="%")
df_d153.head(5)

In [ ]:
check_coverage(df_d153)

In [ ]:
load(df_d153)

## D154

In [ ]:
raw_d154 = read_prepared("D154")

raw_d154.head(5)

In [ ]:
df_d154 = to_contract(raw_d154, "D154", labels="Index")
df_d154.head(5)

In [ ]:
check_coverage(df_d154)

In [ ]:
load(df_d154)

## D155

In [ ]:
raw_d155 = read_prepared("D155")

raw_d155.head(5)

In [ ]:
df_d155 = to_contract(raw_d155, "D155", labels="%")
df_d155.head(5)

In [ ]:
check_coverage(df_d155)

In [ ]:
load(df_d155)

## D156

In [ ]:
raw_d156 = read_prepared("D156")

raw_d156.head(5)

In [ ]:
df_d156 = to_contract(raw_d156, "D156", labels="% of GDP")
df_d156.head(5)

In [ ]:
check_coverage(df_d156)

In [ ]:
load(df_d156)

## D157

In [ ]:
raw_d157 = read_prepared("D157")

raw_d157.head(5)

In [ ]:
df_d157 = to_contract(raw_d157, "D157", labels="%")
df_d157.head(5)

In [ ]:
check_coverage(df_d157)

In [ ]:
load(df_d157)

## D158Trade concentration by geopolitical bloc (HHI, 0-10000). Not a directly-published Comtrade series —derived from the same cached bilateral extract used for D153 (reporter x partner x year, 2000-2024,`cmdCode=TOTAL`, flows M+X). Each country's two-way trade is split into three groups (West bloc,China+Russia bloc, rest of world) and HHI'd, same convention as D163. Built 2026-08-17 to fill the gapflagged in the Topic 5 architecture; see `MODEL_LIMITATIONS.md` and Topic 5 Appendix B.The derivation cell below is idempotent and writes straight to `Prepared/D158.csv` — re-running itjust reproduces the same file. Skip it and go straight to `read_prepared("D158")` if the file alreadyexists.

In [ ]:
# One-time derivation: raw bilateral JSON -> Prepared/D158.csv.# Only needs to run once; Prepared/D158.csv is already checked in.D158_SRC = RAW_DIR / "Topic 3-5" / "UN Comtrade" / "Original" / "Derived Proxies" / "D153_intraregional_total.json"M49_TO_ISO3 = {    32: "ARG", 36: "AUS", 50: "BGD", 76: "BRA", 124: "CAN", 156: "CHN", 180: "COD",    231: "ETH", 251: "FRA", 276: "DEU", 360: "IDN", 364: "IRN", 376: "ISR", 380: "ITA",    392: "JPN", 398: "KAZ", 404: "KEN", 410: "KOR", 484: "MEX", 528: "NLD", 566: "NGA",    586: "PAK", 616: "POL", 643: "RUS", 682: "SAU", 699: "IND", 704: "VNM", 710: "ZAF",    784: "ARE", 792: "TUR", 804: "UKR", 818: "EGY", 826: "GBR", 842: "USA",}WEST_BLOC = {"USA", "CAN", "GBR", "FRA", "DEU", "ITA", "NLD", "POL", "JPN", "KOR", "AUS"}CHINA_RUSSIA_BLOC = {"CHN", "RUS"}west_codes = {k for k, v in M49_TO_ISO3.items() if v in WEST_BLOC}cr_codes = {k for k, v in M49_TO_ISO3.items() if v in CHINA_RUSSIA_BLOC}with open(D158_SRC) as f:    bilateral_d158 = pd.DataFrame(json.load(f))[["reporterCode", "partnerCode", "refYear", "primaryValue"]]bilateral_d158["reporter"] = bilateral_d158["reporterCode"].map(M49_TO_ISO3)assert bilateral_d158["reporter"].notna().all(), "unmapped reporter code in D153/D158 source"bilateral_d158 = bilateral_d158[bilateral_d158["partnerCode"] != 0]                      # drop World aggregatebilateral_d158 = bilateral_d158[bilateral_d158["reporterCode"] != bilateral_d158["partnerCode"]]  # drop self-tradedef _bloc(code):    if code in west_codes: return "West"    if code in cr_codes: return "ChinaRussia"    return "Rest"bilateral_d158["bloc"] = bilateral_d158["partnerCode"].map(_bloc)agg_d158 = (bilateral_d158.groupby(["reporter", "refYear", "bloc"])["primaryValue"]            .sum().unstack("bloc", fill_value=0.0))for col in ["West", "ChinaRussia", "Rest"]:    if col not in agg_d158.columns:        agg_d158[col] = 0.0agg_d158["total"] = agg_d158[["West", "ChinaRussia", "Rest"]].sum(axis=1)agg_d158 = agg_d158[agg_d158["total"] > 0]shares_d158 = agg_d158[["West", "ChinaRussia", "Rest"]].div(agg_d158["total"], axis=0) * 100hhi_d158 = (shares_d158 ** 2).sum(axis=1)prepared_d158 = hhi_d158.reset_index()prepared_d158.columns = ["market", "year", "value"]prepared_d158["labels"] = "HHI (0-10000)"prepared_d158["metric"] = ""prepared_d158["series_type"] = "historical"prepared_d158["scenario"] = ""prepared_d158 = prepared_d158.sort_values(["market", "year"]).reset_index(drop=True)prepared_d158.to_csv(PREPARED_DIR / "D158.csv", index=False)print(f"wrote {len(prepared_d158)} rows -> {PREPARED_DIR / 'D158.csv'}")prepared_d158.head(5)

In [ ]:
raw_d158 = read_prepared("D158")raw_d158.head(5)

In [ ]:
df_d158 = to_contract(raw_d158, "D158", labels="HHI (0-10000)")df_d158.head(5)

In [ ]:
check_coverage(df_d158)

In [ ]:
load(df_d158)

## D159

In [ ]:
raw_d159 = read_prepared("D159")

raw_d159.head(5)

In [ ]:
df_d159 = to_contract(raw_d159, "D159", labels="%")
df_d159.head(5)

In [ ]:
check_coverage(df_d159)

In [ ]:
load(df_d159)

## D160

In [ ]:
raw_d160 = read_prepared("D160")

raw_d160.head(5)

In [ ]:
df_d160 = to_contract(raw_d160, "D160", labels="Index")
df_d160.head(5)

In [ ]:
check_coverage(df_d160)

In [ ]:
load(df_d160)

## D161

In [ ]:
raw_d161 = read_prepared("D161")

raw_d161.head(5)

In [ ]:
df_d161 = to_contract(raw_d161, "D161", labels="%")
df_d161.head(5)

In [ ]:
check_coverage(df_d161)

In [ ]:
load(df_d161)

## D162

In [ ]:
raw_d162 = read_prepared("D162")

raw_d162.head(5)

In [ ]:
df_d162 = to_contract(raw_d162, "D162", labels="%")
df_d162.head(5)

In [ ]:
check_coverage(df_d162)

In [ ]:
load(df_d162)

## D163

In [ ]:
raw_d163 = read_prepared("D163")

raw_d163.head(5)

In [ ]:
df_d163 = to_contract(raw_d163, "D163", labels="Index")
df_d163.head(5)

In [ ]:
check_coverage(df_d163)

In [ ]:
load(df_d163)

## D164

In [ ]:
raw_d164 = read_prepared("D164")

raw_d164.head(5)

In [ ]:
df_d164 = to_contract(raw_d164, "D164", labels="%")
df_d164.head(5)

In [ ]:
check_coverage(df_d164)

In [ ]:
load(df_d164)

## D165

In [ ]:
raw_d165 = read_prepared("D165")

raw_d165.head(5)

In [ ]:
df_d165 = to_contract(raw_d165, "D165", labels="%")
df_d165.head(5)

In [ ]:
check_coverage(df_d165)

In [ ]:
load(df_d165)

## D166

In [ ]:
raw_d166 = read_prepared("D166")

raw_d166.head(5)

In [ ]:
df_d166 = to_contract(raw_d166, "D166", labels="% of GDP")
df_d166.head(5)

In [ ]:
check_coverage(df_d166)

In [ ]:
load(df_d166)

## D167

In [ ]:
raw_d167 = read_prepared("D167")

raw_d167.head(5)

In [ ]:
df_d167 = to_contract(raw_d167, "D167", labels="Index")
df_d167.head(5)

In [ ]:
check_coverage(df_d167)

In [ ]:
load(df_d167)

## D168

In [ ]:
raw_d168 = read_prepared("D168")

raw_d168.head(5)

In [ ]:
df_d168 = to_contract(raw_d168, "D168", labels="%")
df_d168.head(5)

In [ ]:
check_coverage(df_d168)

In [ ]:
load(df_d168)

## D169

In [ ]:
raw_d169 = read_prepared("D169")

raw_d169.head(5)

In [ ]:
df_d169 = to_contract(raw_d169, "D169", labels="% of GDP")
df_d169.head(5)

In [ ]:
check_coverage(df_d169)

In [ ]:
load(df_d169)

## D170

In [ ]:
raw_d170 = read_prepared("D170")

raw_d170.head(5)

In [ ]:
df_d170 = to_contract(raw_d170, "D170", labels="%")
df_d170.head(5)

In [ ]:
check_coverage(df_d170)

In [ ]:
load(df_d170)

## D171

In [ ]:
raw_d171 = read_prepared("D171")

raw_d171.head(5)

In [ ]:
df_d171 = to_contract(raw_d171, "D171", labels="No. of months")
df_d171.head(5)

In [ ]:
check_coverage(df_d171)

In [ ]:
load(df_d171)

## D172

In [ ]:
raw_d172 = read_prepared("D172")

raw_d172.head(5)

In [ ]:
df_d172 = to_contract(raw_d172, "D172", labels="% of GDP")
df_d172.head(5)

In [ ]:
check_coverage(df_d172)

In [ ]:
load(df_d172)

## D173

In [ ]:
raw_d173 = read_prepared("D173")

raw_d173.head(5)

In [ ]:
df_d173 = to_contract(raw_d173, "D173", labels="%")
df_d173.head(5)

In [ ]:
check_coverage(df_d173)

In [ ]:
load(df_d173)

## D174

In [ ]:
raw_d174 = read_prepared("D174")

raw_d174.head(5)

In [ ]:
df_d174 = to_contract(raw_d174, "D174", labels="% of GDP")
df_d174.head(5)

In [ ]:
check_coverage(df_d174)

In [ ]:
load(df_d174)

## D175

In [ ]:
raw_d175 = read_prepared("D175")

raw_d175.head(5)

In [ ]:
df_d175 = to_contract(raw_d175, "D175", labels="%")
df_d175.head(5)

In [ ]:
check_coverage(df_d175)

In [ ]:
load(df_d175)

## D176

In [ ]:
raw_d176 = read_prepared("D176")

raw_d176.head(5)

In [ ]:
df_d176 = to_contract(raw_d176, "D176", labels="%")
df_d176.head(5)

In [ ]:
check_coverage(df_d176)

In [ ]:
load(df_d176)

## D177

In [ ]:
raw_d177 = read_prepared("D177")

raw_d177.head(5)

In [ ]:
df_d177 = to_contract(raw_d177, "D177", labels="%")
df_d177.head(5)

In [ ]:
check_coverage(df_d177)

In [ ]:
load(df_d177)

## D178

In [ ]:
raw_d178 = read_prepared("D178")

raw_d178.head(5)

In [ ]:
df_d178 = to_contract(raw_d178, "D178", labels="%")
df_d178.head(5)

In [ ]:
check_coverage(df_d178)

In [ ]:
load(df_d178)

## D180

In [ ]:
raw_d180 = read_prepared("D180")

raw_d180.head(5)

In [ ]:
df_d180 = to_contract(raw_d180, "D180", labels="USD per capita")
df_d180.head(5)

In [ ]:
check_coverage(df_d180)

In [ ]:
load(df_d180)

## D181

In [ ]:
raw_d181 = read_prepared("D181")

raw_d181.head(5)

In [ ]:
df_d181 = to_contract(raw_d181, "D181", labels="% of GDP")
df_d181.head(5)

In [ ]:
check_coverage(df_d181)

In [ ]:
load(df_d181)

## D182

In [ ]:
raw_d182 = read_prepared("D182")

raw_d182.head(5)

In [ ]:
df_d182 = to_contract(raw_d182, "D182", labels="% of GDP")
df_d182.head(5)

In [ ]:
check_coverage(df_d182)

In [ ]:
load(df_d182)

## D183a

In [ ]:
raw_d183a = read_prepared("D183a")

raw_d183a.head(5)

In [ ]:
df_d183a = to_contract(raw_d183a, "D183a", labels="%")
df_d183a.head(5)

In [ ]:
check_coverage(df_d183a)

In [ ]:
load(df_d183a)